# 🛡️ Taivium: Deterministic Privacy for LLM Applications

Welcome to the Taivium demo! This notebook showcases how Taivium protects sensitive data in LLM workflows **without breaking reasoning, context, or output quality**.

🔗 [GitHub Repo](https://github.com/taivium-ai/taivium)

---

## Why Use Taivium?

- **Prevents data leaks** to LLMs
- **Preserves identity and context** (no broken prompts)
- **Deterministic pseudonyms** for consistent anonymization
- **Easy integration** with OpenAI-compatible clients

---

## 🔥 Quick Start: Anonymize Text Instantly

In [1]:
from taivium.engine import Taivium

pipeline = Taivium()

text = "Alice Johnson from Acme Corp emailed alice@acme.com"
result = pipeline.process(text)

print('Original:', result['original'])
print('Anonymized:', result['anonymized'])

Original: Alice Johnson from Acme Corp emailed alice@acme.com
Anonymized: PERSON_709cc2b3a716 from ORG_faf6d0f76d9c emailed EMAIL_ae0b483ad1e4


### Example Output
```
Original: Alice Johnson from Acme Corp emailed alice@acme.com
Anonymized: PERSON_xxxx from ORG_xxxx emailed EMAIL_xxxx
```

You can restore the original text using the mapping:

In [2]:
print('Mapping:', result['mapping'])

Mapping: {'PERSON_709cc2b3a716': {'text': 'Alice Johnson', 'label': 'PERSON', 'source': 'canonical', 'evidence_sources': ('spacy',), 'confidence': 0.75, 'risk': <RiskLevel.MEDIUM: 'medium'>, 'action': <PolicyAction.ANONYMIZE: 'anonymize'>, 'reason': <PolicyDecisionReason.EXPLICIT: 'explicit_rule'>}, 'ORG_faf6d0f76d9c': {'text': 'Acme Corp', 'label': 'ORG', 'source': 'canonical', 'evidence_sources': ('spacy',), 'confidence': 0.75, 'risk': <RiskLevel.MEDIUM: 'medium'>, 'action': <PolicyAction.ANONYMIZE: 'anonymize'>, 'reason': <PolicyDecisionReason.EXPLICIT: 'explicit_rule'>}, 'EMAIL_ae0b483ad1e4': {'text': 'alice@acme.com', 'label': 'EMAIL', 'source': 'canonical', 'evidence_sources': ('regex',), 'confidence': 0.9, 'risk': <RiskLevel.HIGH: 'high'>, 'action': <PolicyAction.ANONYMIZE: 'anonymize'>, 'reason': <PolicyDecisionReason.EXPLICIT: 'explicit_rule'>}}


---
## 🏷️ What Entities Get Detected?

| Entity Type | Example |
|-------------|---------|
| PERSON      | Alice Johnson |
| ORG         | Acme Corp |
| EMAIL       | alice@acme.com |
| PHONE       | +1 415-555-1234 |
| API_KEY     | sk-xxxx |
| LOCATION    | New York |

---

## ⚙️ Custom Policies
You can control how each entity type is handled (allow, anonymize, block):

In [3]:
from taivium.engine import PolicyEngine, PolicyRule, PolicyAction, RiskLevel

policy_engine = PolicyEngine(policy_table={
    "API_KEY": PolicyRule("API_KEY", PolicyAction.BLOCK, RiskLevel.CRITICAL),
    "LOCATION": PolicyRule("LOCATION", PolicyAction.ALLOW, RiskLevel.LOW),
})

pipeline = Taivium(policy_engine=policy_engine)
result = pipeline.process("My API key is sk-1234567890 and I live in New York.")
print('Anonymized:', result['anonymized'])

Blocked sensitive entity: label=API_KEY, id=API_KEY_eae9215d5e81


ValueError: Blocked sensitive entity: API_KEY (id=API_KEY_eae9215d5e81)

---
## 🗄️ Redis Persistence (Optional)
Persist entity mappings across sessions using Redis:

In [4]:
from taivium.session_store import RedisSessionStore

store = RedisSessionStore(
    session_id="user-123",
    redis_url="redis://localhost:6379",
)
pipeline = Taivium(session_store=store)
# Now entity mappings are shared across calls in this session.

---
## 🤖 OpenAI-Compatible Client
Taivium can be used as a drop-in replacement for OpenAI's client:

In [ ]:
from taivium.client import PrivacyClient
import os

client = PrivacyClient(api_key=os.environ.get("OPENAI_API_KEY", "sk-..."))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": "John Doe's email is john@acme.com. Draft a follow-up."
    }],
    deid_response=True,
)
print(response.choices[0].message.content)

---
## 🧠 How It Works

1. **Detection Layer:** spaCy, regex, transformer, LLM
2. **Span Canonicalization:** One entity per span
3. **Identity Engine:** Deterministic pseudonyms
4. **Policy Engine:** ALLOW / ANONYMIZE / BLOCK
5. **Session Store:** In-memory or Redis

```
engine.py
├── collect_evidence()
├── canonicalize_spans()
├── IdentityEngine.resolve()
├── PolicyEngine.evaluate()
├── transform()
└── session_store
```

---
## 📚 More Examples
- See the [examples/](./) folder for more advanced usage and custom detectors.

---
## 📝 License & Support
- MIT License
- Open an issue on GitHub for bugs, questions, or feature requests.

---

✨ **Taivium: Privacy without compromise.**